# 🥐 範例 10：烘焙生鮮—每日出爐備料需求預估 (迴歸篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本教材專為**「烘焙坊店長、主廚與生鮮熟食主管（無程式背景）」**設計。
採用**「一段保證可執行的 Python 程式碼 ＋ 一段 AI 伴學互動提問指引」**的雙軌架構：
- 每一段程式碼都經過嚴謹測試，保證直接點擊即可順暢執行。
- 執行完程式碼後，**下方的文字儲存格**會以烘焙排程白話解釋程式在做什麼，並提供專屬的 **AI 提問 Prompt（提示詞）**，你可以複製貼給 Copilot、ChatGPT 或 Claude，讓 AI 替你深度解說商業決策邏輯！

---

## 📊 一、烘焙門市資料集介紹 (`bakery_demand.csv`)

本數據集記錄了烘焙連鎖門市歷年 1,200 筆商品銷售、天氣與實際出爐需求數據：

| 欄位名稱 | 資料型態 | 數值範例 / 類別 | 烘焙門市業務實務意涵 |
| :--- | :--- | :--- | :--- |
| **烘焙品項** | 類別型 (`object`) | 招牌法式可頌、海鹽奶油生吐司、波士頓生乳捲等 | 主力麵包甜點品項 |
| **星期幾** | 類別型 (`object`) | 星期一 到 星期日 | 週間循環客流 |
| **天氣狀況** | 類別型 (`object`) | 晴朗多雲、晴朗高溫、局部陣雨、大雨冷冽 | 天氣影響客流量 |
| **最高氣溫_度C** | 數值型 (`float64`) | 25.0 (度) | 氣溫影響發酵與甜點偏好 |
| **前一日實際銷量_個** | 數值型 (`int64`) | 180 (個) | 昨天實際賣出的銷量（近期真實動能） |
| **過去一週平均銷量_個** | 數值型 (`int64`) | 175 (個) | 中期平穩基線水準 |
| **當日是否商圈特賣** | 類別型 (`object`) | 是、否 | 百貨促銷檔期加成 |
| **預估次日備料需求量_個 (目標 y)** | 數值型 (`int64`) | **215 (個)** | **預測目標：次日建議出爐顆數（連續數值迴歸）** |

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與繁中字型**：確保圖表顯示中文，避免方塊亂碼 (`□□`)。
2. **資料載入與探索 (EDA)**：檢視品項銷量分佈與天候關聯。
3. **特徵相關性與視覺化**：繪製前日銷量 vs 次日需求之散佈趨勢圖。
4. **特徵工程與 Pipeline 建置**：8:2 切分訓練與測試集，嚴防資料洩漏。
5. **模型選型與 5 折交叉驗證**：比對「線性迴歸」與「隨機森林迴歸器」。
6. **測試集大考與排盤策略**：評估 R²、MAE，輸出模型並模擬次日備料。

---  
### 【步驟 0】：環境準備與繁體中文字型設定

In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！準備進入烘焙備料銷量預測。")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入數據分析庫並載入繁體中文字型庫，確保銷量散佈圖軸標籤正常顯示。

---
💡 **AI 提問小抄：**

1. 📌 **時序滯後特徵 (Lag Features)**
   > 「在烘焙銷量預測中，為什麼『前一日銷量』與『上週同期銷量』是最有力量的預測特徵？這在機器學習時間序列特徵工程中叫什麼？」


---  
### 【步驟 1】：載入資料與烘焙銷量結構探索 (EDA)

In [ ]:
# [步驟 1 程式碼] 讀取 bakery_demand.csv 並檢視資料概況
df = pd.read_csv("bakery_demand.csv")
print("資料集維度（列, 欄）：", df.shape)
print("\n--- 資料前 5 筆預覽 ---")
display(df.head())
print("\n--- 烘焙品項出爐需求摘要統計 ---")
display(df.groupby("烘焙品項")["預估次日備料需求量_個"].describe().round(1))

#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  讀取 1,200 筆烘焙紀錄，依據 5 大品項查看需求顆數的平均值與分佈（如法式可頌平均 180 個、生乳捲約 60 個）。

---
💡 **AI 提問小抄：**

1. 📌 **品項銷量異質性**
   > 「不同品項（如常溫可頌 vs 冷藏蛋糕）的銷量體量相差 3 倍。在同一個模型中訓練，是否會導致大品項主導了模型的損失函數？該如何處理？」


---  
### 【步驟 2】：前日銷量與次日需求之散佈趨勢圖繪製

In [ ]:
# [步驟 2 程式碼] 繪製前日銷量 vs 次日需求量散佈圖與星期別箱型圖
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df, x="前一日實際銷量_個", y="預估次日備料需求量_個", hue="烘焙品項", alpha=0.7, ax=axes[0])
axes[0].set_title("前一日實際銷量 vs 次日備料需求量 (依品項區分)")
axes[0].set_xlabel("前一日實際銷量 (個)")
axes[0].set_ylabel("次日備料需求量 (個)")

day_order = ["星期一", "星期二", "星期三", "星期四", "星期五", "星期六", "星期日"]
sns.boxplot(data=df, x="星期幾", y="預估次日備料需求量_個", order=day_order, palette="Pastel1", ax=axes[1])
axes[1].set_title("一週七天出爐銷量分佈")
axes[1].set_ylabel("")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  散佈圖呈現強烈的正向線性走勢（昨天賣得好，今天通常也維持高水準），並透過箱型圖看出週末出爐量顯著提升。

---
💡 **AI 提問小抄：**

1. 📌 **前日效應與突發事件**
   > 「如果前一天突然因為暴雨導致銷量腰斬，第二天卻天氣大晴，模型會不會因為『昨天賣很爛』就盲目下調今天的出爐量？該如何透過『週平均』來穩住基線？」


---  
### 【步驟 3】：特徵工程與 Pipeline 建構

In [ ]:
# [步驟 3 程式碼] 切分訓練/測試集，並建立前處理管線
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=["預估次日備料需求量_個"])
y = df["預估次日備料需求量_個"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = ["最高氣溫_度C", "前一日實際銷量_個", "過去一週平均銷量_個"]
cat_cols = ["烘焙品項", "星期幾", "天氣狀況", "當日是否商圈特賣"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
])

print(f"✅ 資料分割完成！訓練集：{X_train.shape[0]} 筆，測試集：{X_test.shape[0]} 筆")

#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將數據拆分並標準化數值銷量，將品項與星期獨熱編碼封裝進 Pipeline。

---
💡 **AI 提問小抄：**

1. 📌 **防資料洩漏與前置備料時間**
   > 「麵團通常前一天下午就要打好發酵。這意味著我們只能使用到『今天下午以前』的數據，絕對不能使用明天才發生的事後特徵。」


---  
### 【步驟 4】：模型選型與 5 折交叉驗證 (線性迴歸 vs 隨機森林)

In [ ]:
# [步驟 4 程式碼] 使用 5 折交叉驗證評估線性迴歸與隨機森林
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

pipe_lr = Pipeline([("preprocessor", preprocessor), ("regressor", LinearRegression())])
pipe_rf = Pipeline([("preprocessor", preprocessor), ("regressor", RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42))])

scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=5, scoring="r2")
scores_rf = cross_val_score(pipe_rf, X_train, y_train, cv=5, scoring="r2")

print(f"線性迴歸 5 折交叉驗證 平均 R²: {scores_lr.mean():.4f} (±{scores_lr.std():.4f})")
print(f"隨機森林 5 折交叉驗證 平均 R²: {scores_rf.mean():.4f} (±{scores_rf.std():.4f})")

#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  運用 5 折交叉驗證比較兩種演算法對出爐顆數的解釋力。

---
💡 **AI 提問小抄：**

1. 📌 **非線性特賣加成**
   > 「商圈大特賣通常對『可頌』有 50% 銷量爆發加成，但對『生乳捲』加成不明顯。隨機森林如何捕捉品項與促銷的非線性交互？」


---  
### 【步驟 5】：測試集大考與指標解讀 (MAE, RMSE, R²)

In [ ]:
# [步驟 5 程式碼] 在全新測試集評估模型並繪製預測散佈圖
from sklearn.metrics import r2_score, mean_absolute_error

pipe_rf.fit(X_train, y_train)
y_pred = pipe_rf.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"=== 測試集最終評估結果 ===")
print(f"決定係數 R²: {r2:.4f} (模型解釋了 {r2*100:.1f}% 的出爐需求波動)")
print(f"平均絕對誤差 MAE: {mae:.1f} 個 (平均每品項出爐誤差顆數)")

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color="#d97706")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "k--", lw=2, label="完美預測對角線")
plt.title(f"真實銷量 vs 模型預測出爐需求量 (MAE={mae:.1f}個)")
plt.xlabel("實際銷量需求 (個)")
plt.ylabel("預估出爐備料 (個)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  輸出測試集 MAE 誤差顆數（平均誤差約 10 顆左右），並以散佈圖呈現擬合精度。

---
💡 **AI 提問小抄：**

1. 📌 **MAE 換算為金錢損耗**
   > 「如果單一品項平均誤差為 10 個，一顆可頌成本 25 元。這相當於每天每品項的剩食或缺貨風險約為 250 元。我們該如何設定『安全備料緩衝 (Safety Stock)』？」


---  
### 【步驟 6】：次日出爐顆數即時估算 (Inference) 與烤盤排程

In [ ]:
# [步驟 6 程式碼] 保存模型管線並示範明日招牌可頌出爐估算
joblib.dump(pipe_rf, "model.joblib")
print("✅ 模型管線已成功保存至 model.joblib！")

# 模擬店長今晚估算明天「星期六」招牌法式可頌需求
croissant_plan = pd.DataFrame([{
    "烘焙品項": "招牌法式可頌",
    "星期幾": "星期六",
    "天氣狀況": "晴朗多雲",
    "最高氣溫_度C": 26.0,
    "前一日實際銷量_個": 195,
    "過去一週平均銷量_個": 180,
    "當日是否商圈特賣": "是"
}])

loaded_model = joblib.load("model.joblib")
est_demand = int(loaded_model.predict(croissant_plan)[0])
trays = int(np.ceil(est_demand / 12))

print(f"\n🥐 次日烘焙出爐報告：")
print(f"預估建議出爐量：{est_demand} 個")
print(f"後廚排程：需烘烤 {trays} 烤盤（每盤 12 個），建議 07:30 晨間首批出爐 12 盤，午後 15:30 補烤剩餘盤數！")

#### 💬【步驟 6 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  保存模型並模擬即時推算招牌可頌的出爐顆數與換算標準烤盤數。

---
💡 **AI 提問小抄：**

1. 📌 **產能極限約束 (Capacity Constraint)**
   > 「如果模型預估明天要出爐 350 個，但門市烤箱全天最多只能烤 280 個。在資料科學與運籌學中，該如何將『產能上限約束』整合進預測系統？」
